# Crash recovery for durable runs (`meta.resume`)

The central promise of a **durable** Meta-Evolve run: a process that dies partway through loses no committed work, and once `resume`-d reaches the **exact** result an uninterrupted run would have reached.

**Interrupted is logically equivalent to uninterrupted.**

This notebook walks the lifecycle end to end:
1. Run a durable-local run to completion, uninterrupted, on path **A** — its `best()` is the reference.
2. Start the *same* run on path **B**, but simulate a process crash after a few committed batches.
3. Reopen path B and `resume` the run.
4. Assert the resumed `best()` equals the uninterrupted reference.

Key concepts: **atomic batch commits** (a crash leaves only whole batches, never a half-written one), the **recovery frontier** (`resume` replays the committed prefix and continues deterministically from the last decided step), and **deterministic run identity** (the run id is derived from the inputs, so the run on path B shares path A's id and `resume` can address it).

## Components — imported from the example module
`durable-local` mode requires **registered, source-digestible, module-level** functions so `resume` can verify the components it replays are byte-for-byte the ones recorded. Functions defined inside a notebook cell have **no resolvable source file**, so durable-local would reject them (`CapabilityError`). We therefore import the real components from `examples/operations/crash_recovery/main.py` (an importable module) and show their source below — the notebook orchestrates the lifecycle with the example's own building blocks.

In [1]:
import inspect
import sys
from pathlib import Path

import meta_evolve as meta

# main.py lives next to this notebook; add its dir so its module-level
# (source-digestible) proposer/evaluator/helpers can be imported.
sys.path.insert(0, str(Path('examples/operations/crash_recovery').resolve()))
import main  # the runnable example this notebook mirrors

TARGET = main.TARGET
print(inspect.getsource(main.propose))
print(inspect.getsource(main.evaluate))

def propose(parent: object, context: object) -> object:
    """Climb toward the target by one each step. Deterministic and retry-safe:
    the same parent always yields the same child, so replaying it during
    recovery reproduces the identical candidate.

    ``context`` is deliberately ignored: this is a fixed +1 climb with no
    randomness. (Sibling example 01 instead draws from ``context.rng`` -- that
    is where the per-step context earns its keep.)"""

    return parent + 1

def evaluate(candidate: object) -> float:
    """Higher is better; the search maximizes this score. Closer to the target
    scores higher, and the target itself is the ceiling."""

    return -abs(float(candidate) - TARGET)



In [2]:
# build_components / build_task / start_run are the example's own helpers.
# start_run is the single run shape; the run id derives from its inputs, so the
# same inputs on two durable paths produce the SAME run id (how resume addresses it).
print(inspect.getsource(main.build_components))
print(inspect.getsource(main.start_run))

components = main.build_components()

def build_components() -> Components:
    """Register the example callables and retain their typed references."""
    registry = Registry()
    proposer = registry.register_proposer(
        "example:propose", "1", propose, deterministic=True, retry_safe=True
    )
    evaluator = registry.register_evaluator(
        "example:evaluate",
        "1",
        evaluate,
        deterministic=True,
        retry_safe=True,
        configuration={"target": TARGET},
    )
    return Components(registry, proposer, evaluator)

def start_run(storage: meta.Storage, components: Components) -> meta.Run:
    """The one run shape used everywhere in this example. Because the run id is
    derived from these inputs, calling ``start_run`` with the same components on
    two different durable paths produces runs with the SAME id."""

    experiment = meta.Experiment(
        task=build_task(components.evaluator),
        seed=0,
        proposer=components.proposer,
        registry=components.registry,

## Simulating a crash — *you never write this*
A real crash (power loss, `kill -9`, OOM) needs no code; the OS/hardware supplies it. `CrashAfter` exists **only** so a deterministic notebook can stand in for one — in production you never author it or touch `storage._commit_strategy`.

A faithful crash **never truncates** the append-only store: it just stops committing after some whole batch `n`, so the `n` committed batches on disk are exactly what an uninterrupted run would have written, with no partial batch.

In [3]:
import tempfile

# The example already defines the crash simulation; we reuse it (and show it).
SimulatedCrash = main.SimulatedCrash
install_crash = main._install_crash
print(inspect.getsource(main.CrashAfter))

tmp = Path(tempfile.mkdtemp())
print('workspace:', tmp)

class CrashAfter:
    """Commit-strategy wrapper: perform the first ``n`` commits for real, then
    raise ``SimulatedCrash`` on the next one. Never touches what was committed."""

    def __init__(self, inner: object, *, n: int) -> None:
        self._inner = inner
        self._n = n
        self.commits = 0

    def commit(
        self,
        run_id: RunId,
        expected_version: int,
        *,
        events: Iterable[EventEnvelope],
        occurrences: Iterable[Artifact],
    ) -> int:
        if self.commits >= self._n:
            raise SimulatedCrash(f"simulated process crash after {self._n} committed steps")
        version = self._inner.commit(
            run_id, expected_version, events=events, occurrences=occurrences
        )
        self.commits += 1
        return version

workspace: /var/folders/fn/fg595s6s7wd8d9z3jhr5vsbw0000gn/T/tmp5ituhb2o


## (1) Uninterrupted reference run on path A
A counting wrapper with an unreachable threshold rides along to tally path A's own committed batches.

In [4]:
path_a = tmp / 'path-A'
ref_storage = meta.Storage.durable(path_a)
counter = install_crash(ref_storage, n=10_000)  # never fires; only counts
reference = main.start_run(ref_storage, components)
run_id = reference.id
reference_best = reference.best().value
committed_batches = counter.commits
complete_len = len(meta.Storage.durable(path_a).events.read(run_id))
print('run id          :', run_id)
print('best()          :', reference_best)
print('committed batches:', committed_batches)
print('events on disk  :', complete_len)

run id          : run-72c1e35d855eae5221fd9084ed59dd6027923a87c246dc9c46bde07e1a8a9e99
best()          : 6
committed batches: 23
events on disk  : 49


## (2) Start the same run on path B — then crash mid-way
The run id matches path A's by construction, so we can `resume` it by id.

In [5]:
crash_after = max(1, committed_batches // 2)
assert 1 <= crash_after < committed_batches
path_b = tmp / 'path-B'
crash_storage = meta.Storage.durable(path_b)
crasher = install_crash(crash_storage, n=crash_after)
crashed = False
try:
    main.start_run(crash_storage, components)
except SimulatedCrash as e:
    crashed = True
    print('crashed:', e)
assert crashed and crasher.commits == crash_after
crashed_len = len(meta.Storage.durable(path_b).events.read(run_id))
print(f'events on disk  : {crashed_len}  (a genuine PARTIAL stream vs {complete_len} complete)')

crashed: simulated process crash after 11 committed steps
events on disk  : 25  (a genuine PARTIAL stream vs 49 complete)


## (3) Reopen path B and `resume(run_id, ...)`
No crash wrapper this time — a fresh durable handle on the same path, exactly as a restarted process would.

In [6]:
resumed = meta.resume(run_id, storage=meta.Storage.durable(path_b), registry=components.registry)
resumed_best = resumed.best().value
resumed_len = len(meta.Storage.durable(path_b).events.read(run_id))
print(f'events on disk  : {resumed_len}  (recovered to the full run)')
print('best()          :', resumed_best)

events on disk  : 49  (recovered to the full run)
best()          : 6


## (4) The payoff — interrupted == uninterrupted

In [7]:
assert resumed.id == run_id
assert resumed_best == reference_best
ok = resumed_best == reference_best
print(f'interrupted == uninterrupted: {ok}  ({resumed_best} == {reference_best})')
print(f'partial stream {crashed_len} events -> recovered {resumed_len} events -> best {resumed_best}')

interrupted == uninterrupted: True  (6 == 6)
partial stream 25 events -> recovered 49 events -> best 6


**What this proves.** The crash left a genuine partial durable stream; `resume` replayed the committed prefix, found the recovery frontier, re-drove the deterministic search from exactly there, and reached the identical `best()` — with a byte-identical final event stream. That is the M4 durable-runs guarantee.

The same guarantee holds for runs that use M3 policy-pushed **context** and pull **experience** (see `tests/test_resume_context.py` / `tests/test_resume_experience_faults.py`).

---
# Advanced examples
The basic lifecycle resumed from a single mid-run crash. These show the guarantee is *total* — resume works from **every** committed boundary — and extends to M3 policy-pushed **context** and pull **experience**. All components come from `main.py` / `advanced.py` (importable + source-digestible, as durable-local requires).

In [8]:
from meta_evolve.application.recovery import classify_frontier
from meta_evolve.application.projections import replay_run_projection
from meta_evolve.domain import ExperienceOperationAttempted
from meta_evolve.errors import ExperienceStoreUnavailable
import advanced  # context/experience components next to main.py
# Include both coordinator and audited-reader crash boundaries.
CRASH = (SimulatedCrash, ExperienceStoreUnavailable)
def n_batches(shape, make_components):
    s = meta.Storage.durable(Path(tempfile.mkdtemp()) / 'c')
    counter = install_crash(s, n=10_000)  # unreachable: only counts
    shape(s, make_components())
    return counter.commits
def sweep(shape, make_components, adir):
    """Crash after EVERY committed boundary K; resume each; return (#boundaries,
    all-equivalent?, best, #mid-attempt-rollforwards)."""
    ref_components = make_components()
    ref = shape(meta.Storage.durable(adir / 'ref'), ref_components)
    rid, best = ref.id, ref.best().value
    ev = meta.Storage.durable(adir / 'ref').events.read(rid)
    total = n_batches(shape, make_components)
    all_ok, rollfwd = True, 0
    for k in range(1, total):
        p = adir / f'k{k}'
        # Install and run on one handle; resume from a freshly opened handle.
        s = meta.Storage.durable(p)
        install_crash(s, n=k)
        try:
            shape(s, make_components())
        except CRASH:
            pass
        crashed = meta.Storage.durable(p).events.read(rid)
        if crashed and isinstance(crashed[-1].body, ExperienceOperationAttempted):
            if replay_run_projection(rid, crashed).resolution_for(crashed[-1].body.operation_id) is None:
                rollfwd += 1
        registry = make_components().registry
        r = meta.resume(rid, storage=meta.Storage.durable(p), registry=registry)
        if not (r.best().value == best and meta.Storage.durable(p).events.read(rid) == ev):
            all_ok = False
    return rid, ev, total - 1, all_ok, best, rollfwd

## A. Fault matrix — resume from *every* committed boundary
Crash after each batch K in turn; each resume reaches the identical `best()` and a byte-identical final stream.

In [9]:
adir = Path(tempfile.mkdtemp())
rid_a, ev_a, boundaries, all_ok, best_a, _ = sweep(main.start_run, main.build_components, adir)
print(f'plain run: {boundaries} interior boundaries, all equivalent = {all_ok}, best = {best_a}')

plain run: 22 interior boundaries, all equivalent = True, best = 6


## B. The recovery frontier
`resume` classifies *where* the crash left the run (via `classify_frontier`) before continuing deterministically. The phases seen across boundaries:

In [10]:
phases = {}
total_a = boundaries + 1
for k in range(1, total_a):
    p = adir / f'phase{k}'
    s = meta.Storage.durable(p)  # install + run on one handle (see sweep note)
    install_crash(s, n=k)
    try:
        main.start_run(s, main.build_components())
    except CRASH:
        pass
    events = meta.Storage.durable(p).events.read(rid_a)
    phase = classify_frontier(events, replay_run_projection(rid_a, events)).name
    phases.setdefault(phase, []).append(k)
for name, ks in phases.items():
    print(f'  {name:<18} at boundaries {ks}')

  DECISION_READY     at boundaries [1, 4, 7, 10, 13, 16, 19, 22]
  TRIAL_DECIDED      at boundaries [2, 5, 8, 11, 14, 17, 20]
  EVALUATION_PENDING at boundaries [3, 6, 9, 12, 15, 18, 21]


## C. Cold rebuild from empty tables
A fresh process reopens the durable store; the read model is rebuilt purely from the committed event log — no in-memory state carried over.

In [11]:
cold = meta.Run(rid_a, meta.Storage.durable(adir / 'ref'))
print('best() from a cold store :', cold.best().value)
print('winning lineage steps    :', [t.logical_step for t in cold.lineage()])

best() from a cold store : 6
winning lineage steps    : [0, 1, 2, 3, 4, 5, 6]


## D. Resumable with M3 policy-pushed *context* (`AncestorsOnly`)
The run records a context-policy manifest, its declaration, and each per-trial selection. `resume` rebuilds the context policy from the registry and reuses the committed selections — equivalent at every boundary.

In [12]:
_, _, cb, cok, cbest, _ = sweep(advanced.context_run, advanced.build_components, Path(tempfile.mkdtemp()))
print(f'context run: {cb} boundaries, all equivalent = {cok}, best = {cbest}')

context run: 29 boundaries, all equivalent = True, best = 6


## E. Resumable with M3 pull *experience* (`PullAccess`)
Here the proposer **reads experience** each step, audited as an `attempt` then a `resolution` (two atomic batches). A crash *between* them leaves a trailing unresolved attempt; `resume` **rolls it forward** — recomputing the resolution and reusing the committed operation id — to a byte-identical stream.

In [13]:
_, _, eb, eok, ebest, rollfwd = sweep(advanced.experience_run, advanced.build_components, Path(tempfile.mkdtemp()))
print(f'experience run: {eb} boundaries, all equivalent = {eok}, best = {ebest}')
print(f'  mid-operation crash boundaries rolled forward on resume: {rollfwd}')

experience run: 43 boundaries, all equivalent = True, best = 6
  mid-operation crash boundaries rolled forward on resume: 7


## F. Resume is safe to re-run
Resuming an already-complete run is a no-op returning the same result. (Two processes racing to resume one crashed run is also safe: exactly one commits and the other gets `EventStreamConflict` — never a double-commit or partial write. See `tests/test_resume_faults.py`.)

In [14]:
again_components = main.build_components()
again = meta.resume(rid_a, storage=meta.Storage.durable(adir / 'ref'), registry=again_components.registry)
print('re-resume best()   :', again.best().value)
print('stream unchanged   :', meta.Storage.durable(adir / 'ref').events.read(rid_a) == ev_a)

re-resume best()   : 6
stream unchanged   : True
